# 3차 소스 준비: 개인 기준선·누적 스트레스
Purpose: build the sanitized Phase 3 train/validation source from the private Oracle truth bundle.

기존 검증 저장 순서는 01 → 03 → 02 → 04이며, 그 다음 이 노트북을 Save Version 한 뒤 06을 실행합니다.

> Warning: this is an oracle/sanity-only synthetic-data benchmark. It is not real-device, medical, or locked-test performance evidence.

In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False
RUN_DATA_PREPARATION = True
OUTPUT_ROOT = Path("/kaggle/working/phase3")


def discover_bundle() -> tuple[Path, Path]:
    input_root = Path("/kaggle/input")
    splits = next(input_root.rglob(f"data/registry/{SERIES_ID}/splits.parquet"))
    wheel = next(input_root.rglob("multisensor_ml-*.whl"))
    return splits.parents[2], wheel


if RUN_DATA_PREPARATION:
    data_root, wheel_path = discover_bundle()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(wheel_path)],
        check=True,
    )
    from multisensor_ml.phase3_contracts import Phase3Config
    from multisensor_ml.phase3_pipeline import prepare_phase3_source

    config = Phase3Config(
        schema_version="goal1.5/phase3-personal-pattern/v1",
        series_id=SERIES_ID,
        split_counts=EXPECTED_SPLIT_COUNTS,
        warmup_sec=1800,
        lookback_sec=21600,
        refresh_sec=60,
        weight_caps=(0.0, 0.25, 0.5, 1.0),
        cumulative_horizons_sec=(1800, 21600, 86400, 259200),
        baseline_load_influence_cap=0.10,
        run_locked_test=False,
        raw_root=data_root / "raw",
        outcome_root=data_root / "outcomes",
        registry_root=data_root / "registry",
        artifact_root=OUTPUT_ROOT,
        random_state=20260730,
    )
    source_path = prepare_phase3_source(config)
    shutil.copy2(data_root / "registry" / SERIES_ID / "splits.parquet", OUTPUT_ROOT / "splits.parquet")
    shutil.copy2(wheel_path, OUTPUT_ROOT / wheel_path.name)
    source_manifest = json.loads((OUTPUT_ROOT / "source" / "manifest.json").read_text())
    assert source_manifest["locked_test_read"] is False
    assert source_path.name == "phase3_source.parquet"
    print(json.dumps(source_manifest, indent=2, ensure_ascii=False))